# 고객 리뷰 감성 분석 — OpenAI API · JSON 출력 · 판단 근거 키워드

수업 필기 `고객_리뷰_감성_분석_API` 를 실행 가능하도록 정리 · 개선한 노트입니다.

- 📖 교재 : `AI Agent` 교재10 **p.15~17** (실습: 고객 리뷰 감성 분석 — 개념 · 활용 분야 · 프롬프트 작성 전략 · 예시)
- 🔗 앞 노트 : [`openai_api_basics.ipynb`](./openai_api_basics.ipynb) (API 기초 · JSON) · [`prompt_output_format.ipynb`](./prompt_output_format.ipynb) (JSON 모드 · 형식 지정)
- 📝 원본에서 고친 곳은 그 자리에 **📝 고친 부분** 으로 적고 맨 끝 표에 모았습니다

> **⚠️ 이 노트의 API 셀은 출력이 비어 있습니다.** 필기 원본은 `TypeError` 에서 멈춰 수업 때의 모델 응답이 남아 있지 않고, 정리하는 환경에는 API 키가 없습니다. `# ⏳` 로 시작하는 셀은 `.env` 를 준비하고 **직접 실행**하세요. 표시가 없는 셀(프롬프트 만들기 · JSON 확인 · 오류 재현 · 단어 세기 기준선 · 결과 해석 함수)은 이 노트에서 실제로 실행한 결과입니다.

## 0. 준비

`pip install openai python-dotenv pandas` · `.env` 에 `OPENAI_API_KEY="sk-…"` ([`openai_api_basics.ipynb`](./openai_api_basics.ipynb) 0절)

In [1]:
# 필요한 라이브러리 임포트
import os, json
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# .env → API 키 (키 값은 출력하지 않는다)
print('.env 읽음 :', load_dotenv(find_dotenv(usecwd=True)), '| 키 있음 :', bool(os.getenv('OPENAI_API_KEY')))

# OpenAI 서버와의 연결 통로 생성
client = OpenAI()

.env 읽음 : True | 키 있음 : True


## 1. 감성 분석이란 (교재 p.15)

- **감성 분석(Sentiment Analysis)** : 글에 담긴 감정(긍정 · 부정 · 중립)을 판별하는 일
- 쓰는 곳 : 쇼핑몰 · 배달 앱 **리뷰 분석**(만족도 · 개선점), **SNS 반응**(브랜드 평판 · 마케팅)
- 예전 방식 : 긍정 · 부정으로 표시한 리뷰 수만 개로 분류 모델을 **학습**
- 오늘 방식 : 학습 없이 LLM 에게 **프롬프트로 시킨다** — 분류 기준도, 출력 모양도 글로 정한다

오늘의 리뷰 — 한 글 안에 좋은 말과 아쉬운 말이 섞여 있다.

In [2]:
# 고객 리뷰 텍스트
customer_review = """
바지 밑위가 길고 제 허벅지는 터져나가서 상의를 넣어 입고 가만있을 때 정면모습이 쪼금 애매한 주름이 지네요
그래도 착용감은 편안하고 밝은 바지인데도 부한 느낌도 없어서 마음에 들어요
크림진이 하나도 없어서 고민고민하다 구매했는데 이정도면 만족입니다
다만 허리에 있는 가죽패치가 검정이라 세탁시 이염될까봐 무서운거랑 바지밑단 주름이 좀 자글자글하게 지는 단점은 있어요!
""".strip()

for i, line in enumerate(customer_review.splitlines(), 1):
    print(f'{i}번 문장 ({len(line)}자) : {line[:34]}…')

1번 문장 (60자) : 바지 밑위가 길고 제 허벅지는 터져나가서 상의를 넣어 입고 가…
2번 문장 (41자) : 그래도 착용감은 편안하고 밝은 바지인데도 부한 느낌도 없어서 …
3번 문장 (36자) : 크림진이 하나도 없어서 고민고민하다 구매했는데 이정도면 만족입…
4번 문장 (66자) : 다만 허리에 있는 가죽패치가 검정이라 세탁시 이염될까봐 무서운…


### 1-1. 기준선 — 단어를 세는 방식으로는?

LLM 을 쓰기 전에 가장 단순한 방법을 먼저 해 본다. 긍정 단어와 부정 단어를 **직접 몇 개 정해서** 세는 방식(사전은 이 리뷰를 보고 내가 고른 것이라 예시일 뿐이다).

In [3]:
positive_words = ['편안', '마음에 들', '만족']                   # 내가 고른 긍정 표현
negative_words = ['애매', '터져', '무서', '단점', '자글자글']     # 내가 고른 부정 표현

rows = []
for i, line in enumerate(customer_review.splitlines(), 1):
    pos = [w for w in positive_words if w in line]
    neg = [w for w in negative_words if w in line]
    rows.append({'문장': i, '긍정 단어': ', '.join(pos), '부정 단어': ', '.join(neg), '긍정 수': len(pos), '부정 수': len(neg)})

count_table = pd.DataFrame(rows)
n_pos, n_neg = count_table['긍정 수'].sum(), count_table['부정 수'].sum()
print(f'긍정 {n_pos}개 · 부정 {n_neg}개 → 단어 수로 판정 : {"긍정" if n_pos > n_neg else "부정" if n_neg > n_pos else "중립"}')
count_table

긍정 3개 · 부정 5개 → 단어 수로 판정 : 부정


,문장,긍정 단어,부정 단어,긍정 수,부정 수
0,1,,"애매, 터져",0,2
1,2,"편안, 마음에 들",,2,0
2,3,만족,,1,0
3,4,,"무서, 단점, 자글자글",0,3


단어 수로는 **부정**(3 대 5)이 나온다. 하지만 사람이 읽으면 이 리뷰는 "**이정도면 만족입니다**" 가 결론이고, 부정 표현은 "쪼금", "그래도", "다만 … 단점은 있어요" 처럼 **양보하는 말** 속에 들어 있다. 문장의 무게와 흐름을 읽어야 하는 일 — LLM 에게 맡기는 이유다. (`부한 느낌도 없어서` 처럼 부정어가 붙어 뜻이 뒤집히는 표현도 단어 세기로는 못 잡는다.)

## 2. 실행 함수

In [4]:
SYSTEM_PROMPT = '너는 한글로 된 고객 리뷰에 대한 경력 10년 이상의 우수한 감성 분석 전문가야. JSON 형식으로 응답해.'

def get_completion(prompt, model='gpt-5-nano'):
    """감성 분석용 Chat Completions 호출 — 답은 항상 JSON 문자열"""
    response = client.chat.completions.create(
        model=model,
        messages=[                                            # ← messages (복수형)
            {'role': 'system', 'content': SYSTEM_PROMPT},     # 시스템 프롬프트 : 역할 + "JSON" 이라는 말(JSON 모드의 조건)
            {'role': 'user', 'content': prompt},              # 사용자 프롬프트
        ],
        response_format={'type': 'json_object'},              # JSON 모드
    )
    return response.choices[0].message.content

### 2-1. 필기의 오류 재현 — `Missing required arguments`

필기 노트북에 남아 있는 오류는 **`messages` 를 `message` 로 적었을 때**의 것이다(나중에 함수는 고쳤지만 호출 셀을 다시 실행하지 않아 오류 출력이 남았다).

In [5]:
# 오류가 났던 때의 코드 그대로 : message (단수형)
try:
    client.chat.completions.create(
        model='gpt-5-nano',
        message=[
            {'role': 'system', 'content': '너는 한글로 된 고객 리뷰에 대한 경력 10년 이상의 우수한 감성 분석'},
            {'role': 'user', 'content': '리뷰'},
        ],
        response_format={'type': 'json_object'},
    )
except TypeError as e:
    print('TypeError :', e)

TypeError : Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given


오류 메시지는 "필요한 인자가 빠졌다 — `messages` 와 `model` 이 있어야 한다" 는 뜻이다. `message` 라는 틀린 이름을 지적하는 게 아니라 **없는 것**(`messages`)을 알려 준다. 요청을 보내기 전에 파이썬이 막은 것이라 요금은 없다.

> **📝 고친 부분** — `message=` → `messages=`. 그리고 함수를 고친 뒤에는 **함수 셀과 호출 셀을 둘 다 다시 실행**해야 한다(오류 출력의 `In[15]` 는 고치기 전 함수). 당시 시스템 프롬프트는 `…우수한 감성 분석` 에서 끊겨 있었고 "JSON" 이라는 말도 없었다 — JSON 모드는 프롬프트에 "JSON" 이 있어야 하므로 지금처럼 `…전문가야. JSON 형식으로 응답해.` 로 고친 것이 맞다.

## 3. 감성 분석 (1) — 긍정 / 부정만

### 3-1. 정답 예시를 프롬프트에 넣을 때 — dict 를 그대로 넣으면 JSON 이 아니다

In [6]:
example = {'감성 분석 결과': '긍정'}

# 필기 원본 : f-string 에 dict 를 그대로 넣음
as_written = f'{example}'
print('필기 방식   :', as_written)          # 작은따옴표 — 파이썬 dict 표기
try:
    json.loads(as_written)
except json.JSONDecodeError as e:
    print('  → JSON 으로 읽으면 :', type(e).__name__, '-', e)

# 고침 : json.dumps 로 JSON 문자열을 만들어 넣는다
example_json = json.dumps(example, ensure_ascii=False)
print('고친 방식   :', example_json)        # 큰따옴표 — JSON
print('  → JSON 으로 읽으면 :', json.loads(example_json))

필기 방식   : {'감성 분석 결과': '긍정'}
  → JSON 으로 읽으면 : JSONDecodeError - Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
고친 방식   : {"감성 분석 결과": "긍정"}
  → JSON 으로 읽으면 : {'감성 분석 결과': '긍정'}


> **📝 고친 부분** — 필기의 `<정답 예시>` 는 `{'감성 분석 결과': '긍정'}` 로 **작은따옴표**였다. 모델에게 "JSON 으로 답해" 라고 하면서 JSON 이 아닌 예시를 보여 준 셈이다(JSON 모드가 문법은 맞춰 주지만 예시는 정확한 편이 낫다). `json.dumps(example, ensure_ascii=False)` 로 만든다.

In [7]:
# 사용자 프롬프트 (1)
user_prompt = f"""
triple backticks 로 감싼 한글 리뷰를 분석해서 긍정인지 부정인지 판단하고,
<정답 예시> 와 같은 키를 가진 JSON 형식으로 제시해줘.
"감성 분석 결과" 의 값은 "긍정" 또는 "부정" 중 하나여야 해.

한글 리뷰: ```{customer_review}```

<정답 예시>
{example_json}
</정답 예시>
"""
print(user_prompt)


triple backticks 로 감싼 한글 리뷰를 분석해서 긍정인지 부정인지 판단하고,
<정답 예시> 와 같은 키를 가진 JSON 형식으로 제시해줘.
"감성 분석 결과" 의 값은 "긍정" 또는 "부정" 중 하나여야 해.

한글 리뷰: ```바지 밑위가 길고 제 허벅지는 터져나가서 상의를 넣어 입고 가만있을 때 정면모습이 쪼금 애매한 주름이 지네요
그래도 착용감은 편안하고 밝은 바지인데도 부한 느낌도 없어서 마음에 들어요
크림진이 하나도 없어서 고민고민하다 구매했는데 이정도면 만족입니다
다만 허리에 있는 가죽패치가 검정이라 세탁시 이염될까봐 무서운거랑 바지밑단 주름이 좀 자글자글하게 지는 단점은 있어요!```

<정답 예시>
{"감성 분석 결과": "긍정"}
</정답 예시>



In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
# Chat Completions API 실행 함수 호출
output1 = get_completion(user_prompt)
print(f'LLM 출력:\n{output1}')

result1 = json.loads(output1)                 # JSON 문자열 → dict
print('감성 :', result1['감성 분석 결과'])

## 4. 감성 분석 (2) — 판단 근거 키워드까지 (교재 p.16)

결과만 받으면 **왜** 그렇게 판단했는지 알 수 없다. 긍정 · 부정으로 본 표현을 함께 달라고 하면 결과를 검토할 수 있고, '어떤 점이 좋았나 / 아쉬웠나' 를 모을 수 있다.

### 4-1. 출력 형식 예시가 올바른 JSON 인지 먼저 확인

교재 p.16 의 형식은 설명용 표기(`"긍정" 또는 "부정"`, `...`)가 섞여 있고, 필기는 깔끔한 예시 문자열을 썼다. 둘 다 `json.loads` 에 넣어 본다.

In [8]:
# 교재 p.16 의 형식 표기 (f-string 안이라 {{ }} 로 적혀 있던 것을 풀어 쓴 모양)
format_textbook = """
{
  "감성 분석 결과": "긍정" 또는 "부정",
  "판단 근거 키워드": {
    "긍정": ["키워드1", "키워드2", ...],
    "부정": ["키워드1", "키워드2", ...]
  }
}
"""

# 필기의 형식 예시
format_note = """
{
    "감성 분석 결과": "긍정",
    "판단 근거 키워드":{
        "긍정":["키워드1","키워드2"],
        "부정":["키워드1","키워드2"]
    }

}
"""

for name, text in [('교재 p.16', format_textbook), ('필기', format_note)]:
    try:
        parsed = json.loads(text)
        print(f'{name:<8} : 올바른 JSON  → 키 {list(parsed.keys())}')
    except json.JSONDecodeError as e:
        print(f'{name:<8} : JSON 아님    → {e}')

교재 p.16  : JSON 아님    → Expecting ',' delimiter: line 3 column 20 (char 22)
필기       : 올바른 JSON  → 키 ['감성 분석 결과', '판단 근거 키워드']


- 교재의 표기는 사람이 읽는 **설명**(`또는`, `...`)이라 JSON 이 아니다. 모델은 알아듣지만, 예시와 규칙을 나누는 편이 분명하다 → **예시는 올바른 JSON**, 값의 범위는 **문장으로**
- 필기의 예시는 올바른 JSON 이다. 다만 `"긍정"` 이라고만 적으면 모델이 예시를 따라 늘 긍정으로 기울 수 있으므로 "긍정 또는 부정 중 하나" 를 글로 덧붙인다
- f-string 안에 JSON 을 직접 쓰면 `{ }` 를 `{{ }}` 로 겹쳐 써야 한다(교재 방식). 예시를 **따로 문자열로** 만들고 `{example2}` 로 끼워 넣으면(필기 방식) 겹쳐 쓸 필요가 없다

In [9]:
# 출력 형식 예시 — dict 로 만들고 json.dumps (항상 올바른 JSON)
example2 = json.dumps(
    {'감성 분석 결과': '긍정', '판단 근거 키워드': {'긍정': ['키워드1', '키워드2'], '부정': ['키워드1', '키워드2']}},
    ensure_ascii=False, indent=2)

# 사용자 프롬프트 (2)
user_prompt2 = f"""
다음 고객 리뷰의 내용을 분석하여 긍정 또는 부정으로 감성을 분류하고,
해당 분류의 판단 근거가 되는 핵심 키워드들을 추출하여 JSON 형식으로 제시해 주세요.

고객 리뷰:
```{customer_review}```

출력 형식:
- 아래 <예시> 와 똑같은 키 구조의 JSON 만 출력합니다.
- "감성 분석 결과" 의 값은 "긍정" 또는 "부정" 중 하나입니다.
- 키워드는 리뷰에 실제로 나온 표현을 그대로 씁니다. 해당 표현이 없으면 빈 목록 [] 으로 둡니다.

<예시>
{example2}
</예시>
"""
print(user_prompt2)


다음 고객 리뷰의 내용을 분석하여 긍정 또는 부정으로 감성을 분류하고,
해당 분류의 판단 근거가 되는 핵심 키워드들을 추출하여 JSON 형식으로 제시해 주세요.

고객 리뷰:
```바지 밑위가 길고 제 허벅지는 터져나가서 상의를 넣어 입고 가만있을 때 정면모습이 쪼금 애매한 주름이 지네요
그래도 착용감은 편안하고 밝은 바지인데도 부한 느낌도 없어서 마음에 들어요
크림진이 하나도 없어서 고민고민하다 구매했는데 이정도면 만족입니다
다만 허리에 있는 가죽패치가 검정이라 세탁시 이염될까봐 무서운거랑 바지밑단 주름이 좀 자글자글하게 지는 단점은 있어요!```

출력 형식:
- 아래 <예시> 와 똑같은 키 구조의 JSON 만 출력합니다.
- "감성 분석 결과" 의 값은 "긍정" 또는 "부정" 중 하나입니다.
- 키워드는 리뷰에 실제로 나온 표현을 그대로 씁니다. 해당 표현이 없으면 빈 목록 [] 으로 둡니다.

<예시>
{
  "감성 분석 결과": "긍정",
  "판단 근거 키워드": {
    "긍정": [
      "키워드1",
      "키워드2"
    ],
    "부정": [
      "키워드1",
      "키워드2"
    ]
  }
}
</예시>



In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
output2 = get_completion(user_prompt2)
print(f'LLM 출력:\n{output2}')

### 4-2. 결과 해석 함수 — 모델의 답을 믿기 전에 검사

JSON 모드는 **문법**만 보장한다. 키가 빠지거나 값이 `긍정적` 처럼 조금 다르게 올 수 있으므로 꺼내 쓰기 전에 확인하고, 모델이 댄 키워드가 **리뷰에 정말 있는 말인지**(지어낸 근거가 아닌지)도 본다.

In [10]:
def parse_sentiment(output, review):
    """모델의 JSON 문자열을 검사해서 표로 쓰기 좋은 dict 로 바꾼다."""
    data = json.loads(output)                                   # 문법이 틀리면 여기서 JSONDecodeError
    label = data.get('감성 분석 결과')
    if label not in ('긍정', '부정'):                            # 정해 준 값이 아니면 알려 준다
        raise ValueError(f'예상하지 못한 감성 값 : {label!r}')
    keywords = data.get('판단 근거 키워드', {})
    pos, neg = keywords.get('긍정', []), keywords.get('부정', [])
    made_up = [k for k in pos + neg if k not in review]          # 리뷰에 없는 표현 = 모델이 바꿔 쓰거나 지어낸 것
    return {'감성': label, '긍정 키워드': pos, '부정 키워드': neg, '리뷰에 없는 키워드': made_up}

# 함수가 제대로 도는지 — 내가 손으로 쓴 시험용 답(모델의 답이 아님)으로 확인
test_ok = json.dumps({'감성 분석 결과': '긍정',
                      '판단 근거 키워드': {'긍정': ['착용감은 편안', '이정도면 만족'], '부정': ['이염될까봐', '핏이 별로']}},
                     ensure_ascii=False)
print(parse_sentiment(test_ok, customer_review))

test_bad = json.dumps({'감성 분석 결과': '긍정적'}, ensure_ascii=False)
try:
    parse_sentiment(test_bad, customer_review)
except ValueError as e:
    print('ValueError :', e)

{'감성': '긍정', '긍정 키워드': ['착용감은 편안', '이정도면 만족'], '부정 키워드': ['이염될까봐', '핏이 별로'], '리뷰에 없는 키워드': ['핏이 별로']}
ValueError : 예상하지 못한 감성 값 : '긍정적'


시험용 답에서 `핏이 별로` 는 리뷰에 없는 말이라 '리뷰에 없는 키워드' 로 걸러졌고, `긍정적` 은 정해 준 값이 아니라 오류로 알려 준다. 실제 모델의 답은 아래 셀에서 같은 함수로 확인한다.

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
result2 = parse_sentiment(output2, customer_review)
print('감성          :', result2['감성'])
print('긍정 키워드   :', result2['긍정 키워드'])
print('부정 키워드   :', result2['부정 키워드'])
print('리뷰에 없는 것 :', result2['리뷰에 없는 키워드'])     # 비어 있어야 근거를 지어내지 않은 것

직접 실행해서 볼 것 : ① 1-1 의 단어 세기는 **부정**이었다. 모델은 어느 쪽인가 ② 모델이 고른 키워드와 내가 고른 단어가 얼마나 겹치나 ③ 여러 번 실행해도 판정이 같은가.

## 5. 더 해보기 — 여러 리뷰 · 5단계 분류 (교재 p.16 ②)

교재의 두 번째 전략은 **더 상세한 분류**(매우 긍정 · 긍정 · 중립 · 부정 · 매우 부정)다. 리뷰가 여러 개면 함수로 묶어 반복하고 표로 모은다.

In [11]:
LABELS_5 = ['매우 긍정', '긍정', '중립', '부정', '매우 부정']

def build_prompt_5(review):
    """5단계 감성 분류 프롬프트"""
    example = json.dumps({'감성 분석 결과': '중립', '한 줄 근거': '근거 문장'}, ensure_ascii=False)
    return f"""
다음 고객 리뷰의 감성을 {', '.join(LABELS_5)} 중 하나로 분류하고 근거를 한 문장으로 적어 JSON 으로 제시해 주세요.

고객 리뷰:
```{review}```

<예시>
{example}
</예시>
"""

reviews = [
    customer_review,                                                  # 오늘의 리뷰 (섞인 리뷰)
    '배송도 빠르고 핏이 딱 맞아요. 색깔별로 다 사고 싶네요!',
    '한 번 빨았는데 물이 빠지고 허리가 늘어났어요. 환불하고 싶습니다.',
    '그냥 무난해요. 사진이랑 비슷합니다.',
]
print(build_prompt_5(reviews[1]))


다음 고객 리뷰의 감성을 매우 긍정, 긍정, 중립, 부정, 매우 부정 중 하나로 분류하고 근거를 한 문장으로 적어 JSON 으로 제시해 주세요.

고객 리뷰:
```배송도 빠르고 핏이 딱 맞아요. 색깔별로 다 사고 싶네요!```

<예시>
{"감성 분석 결과": "중립", "한 줄 근거": "근거 문장"}
</예시>



In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
rows = []
for review in reviews:                                   # 리뷰마다 한 번씩 호출 (호출 수만큼 요금)
    data = json.loads(get_completion(build_prompt_5(review)))
    rows.append({'리뷰': review[:24] + '…', '감성': data.get('감성 분석 결과'), '근거': data.get('한 줄 근거')})
pd.DataFrame(rows)

## 📝 고친 부분 정리

| 위치 | 필기 원본 | 고친 내용 |
|---|---|---|
| 함수 | `message = [...]` → `TypeError: Missing required arguments` (출력에 남음) | `messages=` · 함수 셀과 호출 셀을 모두 다시 실행 |
| 함수 | (오류 당시) 시스템 프롬프트가 `…우수한 감성 분석` 에서 끊김 · "JSON" 없음 | `…전문가야. JSON 형식으로 응답해.` (JSON 모드의 조건) |
| 정답 예시 | `example = {"감성 분석 결과":"긍정"}` 을 f-string 에 그대로 → `{'감성 분석 결과': '긍정'}` (작은따옴표, JSON 아님) | `json.dumps(example, ensure_ascii=False)` |
| 프롬프트 (1) | `<정답 예시>` 닫는 태그 없음 · `backicks` 오타(출력에 남음) · 값의 범위 설명 없음 | 닫는 태그 · backticks · "긍정 또는 부정 중 하나" |
| 리뷰 문자열 | `f"""…"""` — 끼워 넣는 값이 없는데 f-string | 일반 문자열 + `.strip()` |
| 감성 분석 (2) | 리뷰와 형식 예시만 만들고 프롬프트 · 호출 없음 (미완성) | 프롬프트 · 호출 · 결과 검사 함수까지 작성 |
| 결과 | 문자열을 `print` 만 | `json.loads` → 값 검사 → 리뷰에 없는 키워드 확인 |
| 환경 설정 | `r"C:\Users\Jinho\work\.env"` 절대 경로 · `#환경 설정` 이 코드 셀 | `find_dotenv()` · 마크다운 제목 |
| 제목 | `# 고객 리뷰 감성 분석 (1)` 은 마크다운인데 `(2)` 는 코드 셀 주석 | 둘 다 마크다운 제목 |
| 교재 p.16 | 형식 예시의 `"긍정" 또는 "부정"` · `...` | 설명용 표기라 JSON 이 아님 → 예시는 올바른 JSON, 범위는 문장으로 |